# 01 — Data Collection from ChEMBL

This notebook demonstrates how to download EGFR inhibitor activity data from ChEMBL
and prepare it for QSAR modelling.

**Target:** EGFR kinase (CHEMBL203)  
**Data source:** ChEMBL REST API  
**Output:** Cleaned CSV with SMILES and pIC50 values

In [ ]:
# Install dependencies (if needed)
# !pip install chembl-webresource-client pandas rdkit

In [ ]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
import warnings
warnings.filterwarnings("ignore")

# Connect to ChEMBL API
activity = new_client.activity
molecule = new_client.molecule

TARGET_CHEMBL_ID = "CHEMBL203"  # EGFR
print(f"Fetching activities for target: {TARGET_CHEMBL_ID}")

In [ ]:
# Fetch IC50 data
res = activity.filter(
    target_chembl_id=TARGET_CHEMBL_ID,
    standard_type="IC50",
    relation="=",
    assay_type="B"
).only(["molecule_chembl_id", "canonical_smiles", "standard_value",
        "standard_units", "pchembl_value", "assay_chembl_id"])

df_raw = pd.DataFrame.from_records(res)
print(f"Raw records: {len(df_raw)}")
df_raw.head()

In [ ]:
# Data cleaning
df = df_raw.copy()

# Keep only nM IC50 values
df = df[df["standard_units"] == "nM"].copy()
df = df.dropna(subset=["canonical_smiles", "standard_value"])
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["standard_value"])

# Remove extreme outliers
df = df[(df["standard_value"] > 0) & (df["standard_value"] < 1e8)]

# Convert IC50 (nM) to pIC50
df["pIC50"] = -np.log10(df["standard_value"] * 1e-9)
df = df[(df["pIC50"] >= 1.0) & (df["pIC50"] <= 12.0)]

# Remove duplicates (keep median pIC50 per compound)
df = df.groupby("canonical_smiles").agg(
    pIC50=("pIC50", "median"),
    n_assays=("assay_chembl_id", "nunique")
).reset_index()

print(f"Clean dataset: {len(df)} compounds")
print(f"pIC50 range: {df['pIC50'].min():.2f} — {df['pIC50'].max():.2f}")
print(f"Mean pIC50: {df['pIC50'].mean():.2f}")
df.head()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# pIC50 distribution
axes[0].hist(df["pIC50"], bins=50, color="#1565C0", alpha=0.7, edgecolor="white")
axes[0].axvline(6.0, color="red", linestyle="--", lw=2, label="Active threshold")
axes[0].set_xlabel("pIC50", fontsize=12)
axes[0].set_ylabel("Count", fontsize=12)
axes[0].set_title(f"pIC50 Distribution (n={len(df):,})", fontsize=12)
axes[0].legend()

# Active/inactive split
n_active = (df["pIC50"] >= 6.0).sum()
n_inactive = len(df) - n_active
axes[1].pie([n_active, n_inactive],
            labels=[f"Active (pIC50≥6)
{n_active:,}", f"Inactive
{n_inactive:,}"],
            colors=["#E53935", "#90CAF9"], autopct="%1.0f%%")
axes[1].set_title("Active/Inactive Split")

plt.suptitle("EGFR Dataset Characteristics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/plots/dataset_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Active: {n_active} ({100*n_active//len(df)}%) | Inactive: {n_inactive} ({100*n_inactive//len(df)}%)")

In [ ]:
# Save cleaned dataset
df.to_csv("../data/processed/chembl_egfr_clean.csv", index=False)
print("Saved: data/processed/chembl_egfr_clean.csv")
print(df.describe())